# Fase 4 v2 — 07: minimap cenital pulido (detecciones visibles)

Sobre el solver bueno (`VideoHomographyLines`, smooth_beta=0.7, sin línea media) dibuja:
- **Máscara SAM3** `green_floor` translúcida (segmentación visible).
- **Cajas YOLO** por clase (`robot` rojo, `orange_ball` naranja, `yellow_zone`, `blue_zone`) + label.
- **IDs del tracker** (`_GreedyTracker`) sobre cada objeto.
- **Overlay de homografía** (rectángulo + círculo).
- **Minimap propio** (`render_field`): robots en **rojo**, balón en **naranja**, con estelas.

Colores de equipo (robot_a/robot_b) quedan pendientes: hoy la config tiene UNA clase
`robot`, así que todos van en rojo. Requiere GPU (SAM3 + YOLO).

In [ ]:
import sys, os, subprocess
from pathlib import Path
from collections import defaultdict, deque
import numpy as np, cv2, decord
REPO = Path.cwd().resolve()
while not (REPO / 'src/core/field_template.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
from src.core.sam3_loader import load_sam3
from src.core.segmentation import detect_classes_in_frame, _load_classes
from src.core.detectors import detect_boxes
from src.core.minimap_pipeline import _largest_mask, _largest_component, _objects_from_boxes, _box_centroid, _GreedyTracker
from src.core.homography import project_points
from src.core import field_template as ft
from src.core import field_landmarks as fl
from src.core.homography_multifeature import VideoHomographyLines
OUT = REPO / 'notebooks/fase_4_homografia/outputs/videos'; OUT.mkdir(parents=True, exist_ok=True)
bundle = load_sam3(); ALL = _load_classes()
GREEN=[c for c in ALL if c['name']=='green_floor']; YOLO=[c for c in ALL if 'yolo_id' in c]
BOXCOL={'robot':(255,0,0),'orange_ball':(255,140,0),'yellow_zone':(255,230,0),'blue_zone':(40,120,255)}  # RGB
print('YOLO:', [c['name'] for c in YOLO])

In [ ]:
TH=np.linspace(0,2*np.pi,60); CIRC=np.stack([fl.CENTER_CIRCLE[0]+fl.CENTER_CIRCLE[2]*np.cos(TH),fl.CENTER_CIRCLE[1]+fl.CENTER_CIRCLE[2]*np.sin(TH)],1)
def c2i(Hi,p): return cv2.perspectiveTransform(np.asarray(p,np.float64).reshape(-1,1,2),Hi).reshape(-1,2)
def draw_overlay(img,H):
    if H is None: return
    Hi=np.linalg.inv(H); rect=[fl.LANDMARK_POINTS[n] for n in ['inner_tl','inner_tr','inner_br','inner_bl']]
    cv2.polylines(img,[c2i(Hi,rect).astype(np.int32)],True,(0,255,0),3,cv2.LINE_AA)
    cv2.polylines(img,[c2i(Hi,CIRC).astype(np.int32)],True,(255,140,0),3,cv2.LINE_AA)
base_field,w2px=ft.render_field(scale=2.2,margin_cm=10.0)
def _bar(mm,x1,y1,x2,y2,col):
    cv2.rectangle(mm,w2px((x1,y1)),w2px((x2,y2)),col,-1); cv2.rectangle(mm,w2px((x1,y1)),w2px((x2,y2)),(255,255,255),1)
def make_minimap(trails,cur):
    mm=base_field.copy()
    # porterias ANCHAS y visibles (amarilla x~0-18 izq, azul x~225-243 der; boca y 55-127)
    _bar(mm,0,55,18,127,(255,230,0)); _bar(mm,225,55,243,127,(40,120,255))
    for oid,pts in trails.items():
        col=(255,0,0) if cur.get(oid,('robot',))[0]=='robot' else (255,140,0)
        ps=[w2px(p) for p in pts]
        for j in range(1,len(ps)): cv2.line(mm,ps[j-1],ps[j],col,2,cv2.LINE_AA)
    for oid,(cls,xy) in cur.items():
        col=(255,0,0) if cls=='robot' else (255,140,0); px=w2px(xy)
        cv2.circle(mm,px,9 if cls=='robot' else 7,col,-1,cv2.LINE_AA); cv2.circle(mm,px,9 if cls=='robot' else 7,(255,255,255),1,cv2.LINE_AA)
    # VERTICAL para igualar la vista cenital portrait: roto 90 CW -> amarilla (x chico) ARRIBA, azul ABAJO
    return cv2.rotate(mm,cv2.ROTATE_90_CLOCKWISE)

p='/workspace/Meta_Glasses/18abril/Camara_superior/IMG_9933.MOV'; vr=decord.VideoReader(p)
idxs=list(range(0,len(vr),3))[:110]; vh=VideoHomographyLines(smooth_beta=0.7); tracker=_GreedyTracker()
trails=defaultdict(lambda: deque(maxlen=40)); tmp=str(OUT/'mmp_tmp.mp4'); wr=None; ow2=oh2=None
for k,i in enumerate(idxs):
    rgb=vr[i].asnumpy().copy()
    gmask=_largest_mask(detect_classes_in_frame(rgb,GREEN,bundle).get('green_floor',[]))
    boxes=detect_boxes(rgb,classes=YOLO,conf=0.25)
    yc=_box_centroid(boxes.get('yellow_zone',[])); bc=_box_centroid(boxes.get('blue_zone',[]))
    if gmask is None: H,st,ov=vh.H,'kept',0.0
    else: H,st,ov=vh.update(cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR),_largest_component(gmask),yc,bc)
    if gmask is not None:
        o=rgb.copy(); o[gmask>0]=(0,200,0); rgb=cv2.addWeighted(o,0.18,rgb,0.82,0)
    for cls,bds in boxes.items():
        col=BOXCOL.get(cls,(200,200,200))
        for bd in bds:
            x1,y1,x2,y2=[int(v) for v in bd.bbox]; cv2.rectangle(rgb,(x1,y1),(x2,y2),col,2)
            cv2.putText(rgb,cls,(x1,y1-6),cv2.FONT_HERSHEY_SIMPLEX,0.6,col,2,cv2.LINE_AA)
    draw_overlay(rgb,H)
    objs=tracker.update(_objects_from_boxes({n:boxes.get(n,[]) for n in ('robot','orange_ball')})); cur={}
    if H is not None and objs:
        feet=np.array([foot for _,_,foot in objs],np.float32); cms=project_points(feet,H)
        for (oid,cls,foot),(x,y) in zip(objs,cms):
            cur[oid]=(cls,(float(x),float(y))); trails[oid].append((float(x),float(y)))
            fx,fy=int(foot[0]),int(foot[1]); cv2.putText(rgb,('R' if cls=='robot' else 'B')+str(oid),(fx-10,fy),cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2,cv2.LINE_AA)
    cv2.putText(rgb,st+' ov='+format(ov,'.2f')+' obj='+str(len(cur)),(20,46),cv2.FONT_HERSHEY_SIMPLEX,1.1,(0,255,0),3,cv2.LINE_AA)
    mm=make_minimap(trails,cur); Hh,Ww=rgb.shape[:2]; mw=int(Ww*0.26); mh=int(mm.shape[0]*mw/mm.shape[1])
    rgb[10:10+mh,Ww-mw-10:Ww-10]=cv2.resize(mm,(mw,mh)); cv2.rectangle(rgb,(Ww-mw-10,10),(Ww-10,10+mh),(255,255,255),2)
    comp=cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR)
    if wr is None: oh,ow=comp.shape[:2]; sc=1100/ow; ow2,oh2=1100,int(oh*sc); wr=cv2.VideoWriter(tmp,cv2.VideoWriter_fourcc(*'mp4v'),12,(ow2,oh2))
    wr.write(cv2.resize(comp,(ow2,oh2)))
wr.release(); fin=str(OUT/'minimap_polish_cenital.mp4')
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',tmp,'-vcodec','libx264','-pix_fmt','yuv420p',fin],check=True); os.remove(tmp)
print('STATS',vh.stats(),'MB',round(os.path.getsize(fin)/1e6,2))